####HoistReassignments

In [0]:
%sql
CREATE OR REPLACE VIEW `p-cdh-delivery`.investorreports.summary_by_agent_vw AS 
WITH LineByLine AS (
  SELECT
    *
  FROM
    `p-cdh-delivery`.investorreports.hoist_linebyline
),
Assignment AS (
  SELECT distinct
    CASE
      WHEN LEFT(ExposureKey, 3) = P.Prefix THEN ExposureKey
      ELSE CONCAT(p.Prefix, ExposureKey)
    END ExposureKey,
    a.CDI,
    a.BusinessDate,
    a.PortfolioName,
    TRIM(
      REPLACE(
        REPLACE(
          REPLACE(REPLACE(REPLACE(Channel, 'Mercury', ''), 'Cell', ''), 'Orbit', ''),
          'Light',
          ''
        ),
        'Pearl',
        ''
      )
    ) AS Channel,
    CASE
      WHEN ExternalCompanyFullName ILIKE '%Ανδρικόπουλος%' THEN 'Andrikopoulos'
      WHEN ExternalCompanyFullName ILIKE '%Κοντόπουλος%' THEN 'Kontopoulos'
      WHEN ExternalCompanyFullName ILIKE '%Μαρακάκης%' THEN 'Marakakis'
      WHEN ExternalCompanyFullName ILIKE '%Μούργελας%' THEN 'Mourgelas'
      WHEN ExternalCompanyFullName ILIKE '%Πάνου%' THEN 'Panou'
      WHEN ExternalCompanyFullName ILIKE '%Πατάκας%' THEN 'Patakas'
      WHEN ExternalCompanyFullName ILIKE '%Σιούφας%' THEN 'Sioufas'
      WHEN ExternalCompanyFullName ILIKE '%Σταυροπούλου%' THEN 'Stavropoulou'
      WHEN ExternalCompanyFullName ILIKE '%Τζιαννίνη%' THEN 'Tziannini'
      WHEN ExternalCompanyFullName ILIKE '%Τσάγκου%' THEN 'Tsagkou'
      WHEN ExternalCompanyFullName ILIKE '%Τσαφαράς%' THEN 'Tsafaras'
      WHEN ExternalCompanyFullName ILIKE '%Φράγκου%' THEN 'Fragkou'
      WHEN ExternalCompanyFullName ILIKE '%Χαρακτινιώτης%' THEN 'Charaktiniotis'
      WHEN ExternalCompanyFullName ILIKE '%Ψύχας%' THEN 'Psychas'
      ELSE ExternalCompanyFullName
    END AS ExternalCompany,
    TRIM(
      REPLACE(
        REPLACE(
          REPLACE(REPLACE(REPLACE(Team, 'Mercury', ''), 'Cell', ''), 'Orbit', ''),
          'Light',
          ''
        ),
        'Pearl',
        ''
      )
    ) AS Team
  from
    `p-cdh`.primary.assignment a
      LEFT JOIN `p-cdh`.primary.portfolio p
        on a.PortfolioName = p.PortfolioName
        and a.BusinessDate = p.BusinessDate
  WHERE
    a.PortfolioName IN ('Mercury', 'Orbit', 'Light', 'Cell', 'Pearl Portfolio A')
    and Channel not in (
      'Venus Interim', 'Symbol SME', 'Customer Care', 'Real Estate', 'SB', 'Venus Collection Agents'
    )
    and SourceSystem = 'qca'
    and AssignmentType = 'Channel Assignment'
),
CA AS (
  SELECT
    l.BusinessDate,
    cast(1 as int) AS RowNumber,
    'CA Team' as Channel,
    NULL as Team,
    NULL AS ExternalCompany,
    l.PortfolioName as PortfolioName,
    l.L3869Flag,
    COUNT(DISTINCT l.CDI) AS NumberAssignedBorrowers,
    SUM(ifnull(TLCatMigrationDate, 0)) AS GBVAssigned,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.CDI
      END
    ) AS NumberWorkedDeptors,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.ExposureKey
      END
    ) AS NumberWorkedLoans,
    SUM(
      CASE
        WHEN WorkedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVWorked,
    COUNT(DISTINCT
      CASE
        WHEN ContactedFlag = 1 THEN l.CDI
      END
    ) AS NumberContactedBorrowers,
    SUM(
      CASE
        WHEN ContactedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVContacted,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowers,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApplied,
    COUNT(DISTINCT
      CASE
        WHEN ApprovalFlag = 1 THEN l.CDI
      END
    ) AS NumberApporvedBorrowers,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApproved,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApproved,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ApprovalFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ApprovalFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApproved,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowersIntra,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVAppliedIntra,
    COUNT(DISTINCT
      CASE
        WHEN
          ApprovalFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberApporvedBorrowersIntra,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApprovedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN
                      ApprovalFlag = 1
                      AND AppliedFlag = 1
                    THEN
                      ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApprovedIntra,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApprovedIntra,
    COUNT(DISTINCT
      CASE
        WHEN ImplementationFlag = 1 THEN l.CDI
      END
    ) AS NumberImplementedBorrowers,
    SUM(
      CASE
        WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplemented,
    COUNT(DISTINCT
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberImplementedBorrowersIntra,
    SUM(
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplementedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutImplemented,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ImplementationFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ImplementationFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidImplemented,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
              END
            ) AS FLOAT
          )
            / CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS PotentialCashRecImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateImplemented,
    CAST(
      SUM(distinct
        CASE
          WHEN ifnull(l.L3869Flag, 0) = 0 THEN ifnull(l.MonthlyPayments, 0)
        END
      ) AS FLOAT
    ) AS TotalPaymentsNoL3869,
    CASE
      WHEN
        (CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)) > 0
      THEN
        (
          CAST(SUM(distinct ifnull(l.MonthlyPayments, 0)) AS FLOAT)
            / CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)
        )
      ELSE 0
    END AS RealisedCashRec,
    CAST(
      ifnull(
        SUM(distinct
          CASE
            WHEN ifnull(l.L3869Flag, 0) = 1 THEN ifnull(l.MonthlyPayments, 0)
          END
        ),
        0
      ) AS FLOAT
    ) AS TotalPaymentsL3869,
    SUM(distinct ifnull(l.MonthlyPayments, 0)) AS TotalPayments
  FROM
    LineByLine l
      LEFT JOIN assignment p
        on p.exposurekey = l.ExposureKey
        and p.CDI = l.CDI
        and p.BusinessDate = l.BusinessDate
  WHERE
    Channel ILIKE '%CA Dialer%'
  GROUP BY
    l.PortfolioName,
    l.L3869Flag,
    l.BusinessDate
  UNION ALL
  SELECT
    l.BusinessDate,
    2 AS RowNumber,
    p.Channel as Channel,
    coalesce(p.Team, p.Channel) as Team,
    NULL AS ExternalCompany, --p.ExternalCompany,
    l.PortfolioName,
    l.L3869Flag,
    COUNT(DISTINCT l.CDI) AS NumberAssignedBorrowers,
    SUM(ifnull(TLCatMigrationDate, 0)) AS GBVAssigned,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.CDI
      END
    ) AS NumberWorkedDeptors,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.ExposureKey
      END
    ) AS NumberWorkedLoans,
    SUM(
      CASE
        WHEN WorkedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVWorked,
    COUNT(DISTINCT
      CASE
        WHEN ContactedFlag = 1 THEN l.CDI
      END
    ) AS NumberContactedBorrowers,
    SUM(
      CASE
        WHEN ContactedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVContacted,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowers,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApplied,
    COUNT(DISTINCT
      CASE
        WHEN ApprovalFlag = 1 THEN l.CDI
      END
    ) AS NumberApporvedBorrowers,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApproved,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApproved,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ApprovalFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ApprovalFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApproved,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowersIntra,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVAppliedIntra,
    COUNT(DISTINCT
      CASE
        WHEN
          ApprovalFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberApporvedBorrowersIntra,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApprovedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN
                      ApprovalFlag = 1
                      AND AppliedFlag = 1
                    THEN
                      ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApprovedIntra,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApprovedIntra,
    COUNT(DISTINCT
      CASE
        WHEN ImplementationFlag = 1 THEN l.CDI
      END
    ) AS NumberImplementedBorrowers,
    SUM(
      CASE
        WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplemented,
    COUNT(DISTINCT
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberImplementedBorrowersIntra,
    SUM(
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplementedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutImplemented,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ImplementationFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ImplementationFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidImplemented,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
              END
            ) AS FLOAT
          )
            / CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS PotentialCashRecImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateImplemented,
    CAST(
      SUM(distinct
        CASE
          WHEN ifnull(l.L3869Flag, 0) = 0 THEN ifnull(l.MonthlyPayments, 0)
        END
      ) AS FLOAT
    ) AS TotalPaymentsNoL3869,
    CASE
      WHEN
        (CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)) > 0
      THEN
        (
          CAST(SUM(distinct ifnull(l.MonthlyPayments, 0)) AS FLOAT)
            / CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)
        )
      ELSE 0
    END AS RealisedCashRec,
    CAST(
      ifnull(
        SUM(distinct
          CASE
            WHEN ifnull(l.L3869Flag, 0) = 1 THEN ifnull(l.MonthlyPayments, 0)
          END
        ),
        0
      ) AS FLOAT
    ) AS TotalPaymentsL3869,
    SUM(distinct ifnull(l.MonthlyPayments, 0)) AS TotalPayments
  FROM
    LineByLine l
      LEFT JOIN assignment p
        on p.exposurekey = l.ExposureKey
        and p.CDI = l.CDI
        and p.BusinessDate = l.BusinessDate
  WHERE
    Channel ILIKE '%CA Dialer%'
  GROUP BY
    p.Channel,
    p.Team,
    -- p.ExternalCompany,
    l.PortfolioName,
    l.L3869Flag,
    l.BusinessDate
),
FA AS (
  SELECT
    l.BusinessDate,
    3 AS RowNumber,
    'FA Team' as Channel,
    NULL as Team,
    NULL AS ExternalCompany,
    l.PortfolioName,
    l.L3869Flag,
    COUNT(DISTINCT l.CDI) AS NumberAssignedBorrowers,
    SUM(ifnull(TLCatMigrationDate, 0)) AS GBVAssigned,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.CDI
      END
    ) AS NumberWorkedDeptors,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.ExposureKey
      END
    ) AS NumberWorkedLoans,
    SUM(
      CASE
        WHEN WorkedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVWorked,
    COUNT(DISTINCT
      CASE
        WHEN ContactedFlag = 1 THEN l.CDI
      END
    ) AS NumberContactedBorrowers,
    SUM(
      CASE
        WHEN ContactedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVContacted,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowers,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApplied,
    COUNT(DISTINCT
      CASE
        WHEN ApprovalFlag = 1 THEN l.CDI
      END
    ) AS NumberApporvedBorrowers,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApproved,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApproved,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ApprovalFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ApprovalFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApproved,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowersIntra,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVAppliedIntra,
    COUNT(DISTINCT
      CASE
        WHEN
          ApprovalFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberApporvedBorrowersIntra,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApprovedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN
                      ApprovalFlag = 1
                      AND AppliedFlag = 1
                    THEN
                      ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApprovedIntra,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApprovedIntra,
    COUNT(DISTINCT
      CASE
        WHEN ImplementationFlag = 1 THEN l.CDI
      END
    ) AS NumberImplementedBorrowers,
    SUM(
      CASE
        WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplemented,
    COUNT(DISTINCT
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberImplementedBorrowersIntra,
    SUM(
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplementedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutImplemented,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ImplementationFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ImplementationFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidImplemented,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
              END
            ) AS FLOAT
          )
            / CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS PotentialCashRecImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateImplemented,
    CAST(
      SUM(distinct
        CASE
          WHEN ifnull(l.L3869Flag, 0) = 0 THEN ifnull(l.MonthlyPayments, 0)
        END
      ) AS FLOAT
    ) AS TotalPaymentsNoL3869,
    CASE
      WHEN
        (CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)) > 0
      THEN
        (
          CAST(SUM(distinct ifnull(l.MonthlyPayments, 0)) AS FLOAT)
            / CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)
        )
      ELSE 0
    END AS RealisedCashRec,
    CAST(
      ifnull(
        SUM(distinct
          CASE
            WHEN ifnull(l.L3869Flag, 0) = 1 THEN ifnull(l.MonthlyPayments, 0)
          END
        ),
        0
      ) AS FLOAT
    ) AS TotalPaymentsL3869,
    SUM(distinct ifnull(l.MonthlyPayments, 0)) AS TotalPayments
  FROM
    LineByLine l
      LEFT JOIN assignment p
        on p.exposurekey = l.ExposureKey
        and p.CDI = l.CDI
        and p.BusinessDate = l.BusinessDate
  WHERE
    Channel ILIKE '%Financial Advisor%'
  GROUP BY
    l.PortfolioName,
    l.L3869Flag,
    l.BusinessDate
  UNION ALL
  SELECT
    l.BusinessDate,
    4 AS RowNumber,
    p.Channel as Channel,
    coalesce(p.Team, p.Channel) as Team,
    NULL AS ExternalCompany, -- p.ExternalCompany,
    l.PortfolioName,
    l.L3869Flag,
    COUNT(DISTINCT l.CDI) AS NumberAssignedBorrowers,
    SUM(ifnull(TLCatMigrationDate, 0)) AS GBVAssigned,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.CDI
      END
    ) AS NumberWorkedDeptors,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.ExposureKey
      END
    ) AS NumberWorkedLoans,
    SUM(
      CASE
        WHEN WorkedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVWorked,
    COUNT(DISTINCT
      CASE
        WHEN ContactedFlag = 1 THEN l.CDI
      END
    ) AS NumberContactedBorrowers,
    SUM(
      CASE
        WHEN ContactedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVContacted,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowers,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApplied,
    COUNT(DISTINCT
      CASE
        WHEN ApprovalFlag = 1 THEN l.CDI
      END
    ) AS NumberApporvedBorrowers,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApproved,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApproved,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ApprovalFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ApprovalFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApproved,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowersIntra,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVAppliedIntra,
    COUNT(DISTINCT
      CASE
        WHEN
          ApprovalFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberApporvedBorrowersIntra,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApprovedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN
                      ApprovalFlag = 1
                      AND AppliedFlag = 1
                    THEN
                      ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApprovedIntra,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApprovedIntra,
    COUNT(DISTINCT
      CASE
        WHEN ImplementationFlag = 1 THEN l.CDI
      END
    ) AS NumberImplementedBorrowers,
    SUM(
      CASE
        WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplemented,
    COUNT(DISTINCT
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberImplementedBorrowersIntra,
    SUM(
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplementedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutImplemented,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ImplementationFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ImplementationFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidImplemented,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
              END
            ) AS FLOAT
          )
            / CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS PotentialCashRecImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateImplemented,
    CAST(
      SUM(distinct
        CASE
          WHEN ifnull(l.L3869Flag, 0) = 0 THEN ifnull(l.MonthlyPayments, 0)
        END
      ) AS FLOAT
    ) AS TotalPaymentsNoL3869,
    CASE
      WHEN
        (CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)) > 0
      THEN
        (
          CAST(SUM(distinct ifnull(l.MonthlyPayments, 0)) AS FLOAT)
            / CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)
        )
      ELSE 0
    END AS RealisedCashRec,
    CAST(
      ifnull(
        SUM(distinct
          CASE
            WHEN ifnull(l.L3869Flag, 0) = 1 THEN ifnull(l.MonthlyPayments, 0)
          END
        ),
        0
      ) AS FLOAT
    ) AS TotalPaymentsL3869,
    SUM(distinct ifnull(l.MonthlyPayments, 0)) AS TotalPayments
  FROM
    LineByLine l
      LEFT JOIN assignment p
        on p.exposurekey = l.ExposureKey
        and p.CDI = l.CDI
        and p.BusinessDate = l.BusinessDate
  WHERE
    Channel ILIKE '%Financial Advisor%'
  GROUP BY
    p.Channel,
    p.Team,
    --  p.ExternalCompany,
    l.PortfolioName,
    l.L3869Flag,
    L.BusinessDate
),
LF (
  SELECT
    l.BusinessDate,
    5 AS RowNumber,
    'External LF' as Channel,
    NULL as Team,
    NULL AS ExternalCompany,
    l.PortfolioName,
    l.L3869Flag,
    COUNT(DISTINCT l.CDI) AS NumberAssignedBorrowers,
    SUM(ifnull(TLCatMigrationDate, 0)) AS GBVAssigned,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.CDI
      END
    ) AS NumberWorkedDeptors,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.ExposureKey
      END
    ) AS NumberWorkedLoans,
    SUM(
      CASE
        WHEN WorkedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVWorked,
    COUNT(DISTINCT
      CASE
        WHEN ContactedFlag = 1 THEN l.CDI
      END
    ) AS NumberContactedBorrowers,
    SUM(
      CASE
        WHEN ContactedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVContacted,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowers,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApplied,
    COUNT(DISTINCT
      CASE
        WHEN ApprovalFlag = 1 THEN l.CDI
      END
    ) AS NumberApporvedBorrowers,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApproved,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApproved,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ApprovalFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ApprovalFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApproved,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowersIntra,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVAppliedIntra,
    COUNT(DISTINCT
      CASE
        WHEN
          ApprovalFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberApporvedBorrowersIntra,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApprovedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN
                      ApprovalFlag = 1
                      AND AppliedFlag = 1
                    THEN
                      ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApprovedIntra,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApprovedIntra,
    COUNT(DISTINCT
      CASE
        WHEN ImplementationFlag = 1 THEN l.CDI
      END
    ) AS NumberImplementedBorrowers,
    SUM(
      CASE
        WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplemented,
    COUNT(DISTINCT
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberImplementedBorrowersIntra,
    SUM(
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplementedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutImplemented,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ImplementationFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ImplementationFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidImplemented,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
              END
            ) AS FLOAT
          )
            / CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS PotentialCashRecImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateImplemented,
    CAST(
      SUM(distinct
        CASE
          WHEN ifnull(l.L3869Flag, 0) = 0 THEN ifnull(l.MonthlyPayments, 0)
        END
      ) AS FLOAT
    ) AS TotalPaymentsNoL3869,
    CASE
      WHEN
        (CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)) > 0
      THEN
        (
          CAST(SUM(distinct ifnull(l.MonthlyPayments, 0)) AS FLOAT)
            / CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)
        )
      ELSE 0
    END AS RealisedCashRec,
    CAST(
      ifnull(
        SUM(distinct
          CASE
            WHEN ifnull(l.L3869Flag, 0) = 1 THEN ifnull(l.MonthlyPayments, 0)
          END
        ),
        0
      ) AS FLOAT
    ) AS TotalPaymentsL3869,
    SUM(distinct ifnull(l.MonthlyPayments, 0)) AS TotalPayments
  FROM
    LineByLine l
      LEFT JOIN assignment p
        on p.exposurekey = l.ExposureKey
        and p.CDI = l.CDI
        and p.BusinessDate = l.BusinessDate
  WHERE
    ExternalCompany IS NOT NULL
  GROUP BY
    l.PortfolioName,
    l.L3869Flag,
    l.BusinessDate
  UNION ALL
  SELECT
    l.BusinessDate,
    6 AS RowNumber,
    NULL as Channel,
    NULL AS Team,
    p.ExternalCompany,
    l.PortfolioName,
    l.L3869Flag,
    COUNT(DISTINCT l.CDI) AS NumberAssignedBorrowers,
    SUM(ifnull(TLCatMigrationDate, 0)) AS GBVAssigned,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.CDI
      END
    ) AS NumberWorkedDeptors,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.ExposureKey
      END
    ) AS NumberWorkedLoans,
    SUM(
      CASE
        WHEN WorkedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVWorked,
    COUNT(DISTINCT
      CASE
        WHEN ContactedFlag = 1 THEN l.CDI
      END
    ) AS NumberContactedBorrowers,
    SUM(
      CASE
        WHEN ContactedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVContacted,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowers,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApplied,
    COUNT(DISTINCT
      CASE
        WHEN ApprovalFlag = 1 THEN l.CDI
      END
    ) AS NumberApporvedBorrowers,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApproved,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApproved,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ApprovalFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ApprovalFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApproved,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowersIntra,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVAppliedIntra,
    COUNT(DISTINCT
      CASE
        WHEN
          ApprovalFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberApporvedBorrowersIntra,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApprovedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN
                      ApprovalFlag = 1
                      AND AppliedFlag = 1
                    THEN
                      ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApprovedIntra,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApprovedIntra,
    COUNT(DISTINCT
      CASE
        WHEN ImplementationFlag = 1 THEN l.CDI
      END
    ) AS NumberImplementedBorrowers,
    SUM(
      CASE
        WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplemented,
    COUNT(DISTINCT
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberImplementedBorrowersIntra,
    SUM(
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplementedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutImplemented,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ImplementationFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ImplementationFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidImplemented,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
              END
            ) AS FLOAT
          )
            / CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS PotentialCashRecImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateImplemented,
    CAST(
      SUM(distinct
        CASE
          WHEN ifnull(l.L3869Flag, 0) = 0 THEN ifnull(l.MonthlyPayments, 0)
        END
      ) AS FLOAT
    ) AS TotalPaymentsNoL3869,
    CASE
      WHEN
        (CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)) > 0
      THEN
        (
          CAST(SUM(distinct ifnull(l.MonthlyPayments, 0)) AS FLOAT)
            / CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)
        )
      ELSE 0
    END AS RealisedCashRec,
    CAST(
      ifnull(
        SUM(distinct
          CASE
            WHEN ifnull(l.L3869Flag, 0) = 1 THEN ifnull(l.MonthlyPayments, 0)
          END
        ),
        0
      ) AS FLOAT
    ) AS TotalPaymentsL3869,
    SUM(distinct ifnull(l.MonthlyPayments, 0)) AS TotalPayments
  FROM
    LineByLine l
      LEFT JOIN assignment p
        on p.exposurekey = l.ExposureKey
        and p.CDI = l.CDI
        and p.BusinessDate = l.BusinessDate
  WHERE
    ExternalCompany IS NOT NULL
  GROUP BY
    p.ExternalCompany,
    l.PortfolioName,
    l.L3869Flag,
    l.BusinessDate
),
unassigned AS (
  SELECT
    l.BusinessDate,
    7 AS RowNumber,
    'Unassigned' as Channel,
    NULL AS Team,
    NULL AS ExternalCompany,
    l.PortfolioName,
    l.L3869Flag,
    COUNT(DISTINCT l.CDI) AS NumberAssignedBorrowers,
    SUM(ifnull(TLCatMigrationDate, 0)) AS GBVAssigned,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.CDI
      END
    ) AS NumberWorkedDeptors,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.ExposureKey
      END
    ) AS NumberWorkedLoans,
    SUM(
      CASE
        WHEN WorkedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVWorked,
    COUNT(DISTINCT
      CASE
        WHEN ContactedFlag = 1 THEN l.CDI
      END
    ) AS NumberContactedBorrowers,
    SUM(
      CASE
        WHEN ContactedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVContacted,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowers,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApplied,
    COUNT(DISTINCT
      CASE
        WHEN ApprovalFlag = 1 THEN l.CDI
      END
    ) AS NumberApporvedBorrowers,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApproved,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApproved,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ApprovalFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ApprovalFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApproved,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowersIntra,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVAppliedIntra,
    COUNT(DISTINCT
      CASE
        WHEN
          ApprovalFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberApporvedBorrowersIntra,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApprovedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN
                      ApprovalFlag = 1
                      AND AppliedFlag = 1
                    THEN
                      ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApprovedIntra,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApprovedIntra,
    COUNT(DISTINCT
      CASE
        WHEN ImplementationFlag = 1 THEN l.CDI
      END
    ) AS NumberImplementedBorrowers,
    SUM(
      CASE
        WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplemented,
    COUNT(DISTINCT
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberImplementedBorrowersIntra,
    SUM(
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplementedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutImplemented,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ImplementationFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ImplementationFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidImplemented,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
              END
            ) AS FLOAT
          )
            / CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS PotentialCashRecImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateImplemented,
    CAST(
      SUM(distinct
        CASE
          WHEN ifnull(l.L3869Flag, 0) = 0 THEN ifnull(l.MonthlyPayments, 0)
        END
      ) AS FLOAT
    ) AS TotalPaymentsNoL3869,
    CASE
      WHEN
        (CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)) > 0
      THEN
        (
          CAST(SUM(distinct ifnull(l.MonthlyPayments, 0)) AS FLOAT)
            / CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)
        )
      ELSE 0
    END AS RealisedCashRec,
    CAST(
      ifnull(
        SUM(distinct
          CASE
            WHEN ifnull(l.L3869Flag, 0) = 1 THEN ifnull(l.MonthlyPayments, 0)
          END
        ),
        0
      ) AS FLOAT
    ) AS TotalPaymentsL3869,
    SUM(distinct ifnull(l.MonthlyPayments, 0)) AS TotalPayments
  FROM
    LineByLine l
      LEFT JOIN assignment p
        on p.exposurekey = l.ExposureKey
        and p.CDI = l.CDI
        and p.BusinessDate = l.BusinessDate
  WHERE
    Channel IS NULL
    AND Team is null
    and ExternalCompany IS NULL
    and ClosedAccountFlag = 'No'
  GROUP BY
    l.PortfolioName,
    l.L3869Flag,
    l.BusinessDate
),
paidoff AS (
  SELECT
    l.BusinessDate,
    8 AS RowNumber,
    'Paid-Off' as Channel,
    NULL AS Team,
    NULL AS ExternalCompany,
    l.PortfolioName,
    l.L3869Flag,
    COUNT(DISTINCT l.CDI) AS NumberAssignedBorrowers,
    SUM(ifnull(TLCatMigrationDate, 0)) AS GBVAssigned,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.CDI
      END
    ) AS NumberWorkedDeptors,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.ExposureKey
      END
    ) AS NumberWorkedLoans,
    SUM(
      CASE
        WHEN WorkedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVWorked,
    COUNT(DISTINCT
      CASE
        WHEN ContactedFlag = 1 THEN l.CDI
      END
    ) AS NumberContactedBorrowers,
    SUM(
      CASE
        WHEN ContactedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVContacted,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowers,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApplied,
    COUNT(DISTINCT
      CASE
        WHEN ApprovalFlag = 1 THEN l.CDI
      END
    ) AS NumberApporvedBorrowers,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApproved,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApproved,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ApprovalFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ApprovalFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApproved,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowersIntra,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVAppliedIntra,
    COUNT(DISTINCT
      CASE
        WHEN
          ApprovalFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberApporvedBorrowersIntra,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApprovedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN
                      ApprovalFlag = 1
                      AND AppliedFlag = 1
                    THEN
                      ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApprovedIntra,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApprovedIntra,
    COUNT(DISTINCT
      CASE
        WHEN ImplementationFlag = 1 THEN l.CDI
      END
    ) AS NumberImplementedBorrowers,
    SUM(
      CASE
        WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplemented,
    COUNT(DISTINCT
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberImplementedBorrowersIntra,
    SUM(
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplementedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutImplemented,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ImplementationFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ImplementationFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidImplemented,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
              END
            ) AS FLOAT
          )
            / CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS PotentialCashRecImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateImplemented,
    CAST(
      SUM(distinct
        CASE
          WHEN ifnull(l.L3869Flag, 0) = 0 THEN ifnull(l.MonthlyPayments, 0)
        END
      ) AS FLOAT
    ) AS TotalPaymentsNoL3869,
    CASE
      WHEN
        (CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)) > 0
      THEN
        (
          CAST(SUM(distinct ifnull(l.MonthlyPayments, 0)) AS FLOAT)
            / CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)
        )
      ELSE 0
    END AS RealisedCashRec,
    CAST(
      ifnull(
        SUM(distinct
          CASE
            WHEN ifnull(l.L3869Flag, 0) = 1 THEN ifnull(l.MonthlyPayments, 0)
          END
        ),
        0
      ) AS FLOAT
    ) AS TotalPaymentsL3869,
    SUM(distinct ifnull(l.MonthlyPayments, 0)) AS TotalPayments
  FROM
    LineByLine l
      LEFT JOIN assignment p
        on p.exposurekey = l.ExposureKey
        and p.CDI = l.CDI
        and p.BusinessDate = l.BusinessDate
  WHERE
    Channel IS NULL
    AND Team is null
    and ExternalCompany IS NULL
    and ClosedAccountFlag = 'Yes'
  GROUP BY
    l.PortfolioName,
    l.L3869Flag,
    l.BusinessDate
),
Total AS (
  SELECT
    l.BusinessDate,
    9 AS RowNumber,
    'Grand Total' as Channel,
    NULL AS Team,
    NULL AS ExternalCompany,
    l.PortfolioName,
    NULL AS L3869Flag,
    COUNT(DISTINCT l.CDI) AS NumberAssignedBorrowers,
    SUM(ifnull(TLCatMigrationDate, 0)) AS GBVAssigned,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.CDI
      END
    ) AS NumberWorkedDeptors,
    COUNT(DISTINCT
      CASE
        WHEN WorkedFlag = 1 THEN l.ExposureKey
      END
    ) AS NumberWorkedLoans,
    SUM(
      CASE
        WHEN WorkedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVWorked,
    COUNT(DISTINCT
      CASE
        WHEN ContactedFlag = 1 THEN l.CDI
      END
    ) AS NumberContactedBorrowers,
    SUM(
      CASE
        WHEN ContactedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVContacted,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowers,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApplied,
    COUNT(DISTINCT
      CASE
        WHEN ApprovalFlag = 1 THEN l.CDI
      END
    ) AS NumberApporvedBorrowers,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApproved,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApproved,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ApprovalFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ApprovalFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApproved,
    CAST(
      SUM(
        CASE
          WHEN ApprovalFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApproved,
    COUNT(DISTINCT
      CASE
        WHEN AppliedFlag = 1 THEN l.CDI
      END
    ) AS NumberAppliedBorrowersIntra,
    SUM(
      CASE
        WHEN AppliedFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVAppliedIntra,
    COUNT(DISTINCT
      CASE
        WHEN
          ApprovalFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberApporvedBorrowersIntra,
    SUM(
      CASE
        WHEN ApprovalFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVApprovedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN
                      ApprovalFlag = 1
                      AND AppliedFlag = 1
                    THEN
                      ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutApprovedIntra,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN
                ApprovalFlag = 1
                AND AppliedFlag = 1
              THEN
                l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ApprovalFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN
                    ApprovalFlag = 1
                    AND AppliedFlag = 1
                  THEN
                    l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidApprovedIntra,
    CAST(
      SUM(
        CASE
          WHEN
            ApprovalFlag = 1
            AND AppliedFlag = 1
          THEN
            ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateApprovedIntra,
    COUNT(DISTINCT
      CASE
        WHEN ImplementationFlag = 1 THEN l.CDI
      END
    ) AS NumberImplementedBorrowers,
    SUM(
      CASE
        WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplemented,
    COUNT(DISTINCT
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          l.CDI
      END
    ) AS NumberImplementedBorrowersIntra,
    SUM(
      CASE
        WHEN
          ImplementationFlag = 1
          AND AppliedFlag = 1
        THEN
          ifnull(TLCatMigrationDate, 0)
      END
    ) AS GBVImplementedIntra,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        1
          - (
            CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
                END
              ) AS FLOAT
            )
              / CAST(
                SUM(
                  CASE
                    WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                  END
                ) AS FLOAT
              )
          )
      ELSE 0
    END AS AverageHaicutImplemented,
    CASE
      WHEN
        CAST(
          COUNT(
            CASE
              WHEN ImplementationFlag = 1 THEN l.ExposureKey
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN CountofInstallments
              END
            ) AS FLOAT
          )
            / CAST(
              COUNT(
                CASE
                  WHEN ImplementationFlag = 1 THEN l.ExposureKey
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS AverageInstallmentsImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
        END
      ) AS FLOAT
    ) AS AmounttobePaidImplemented,
    CASE
      WHEN
        CAST(
          SUM(
            CASE
              WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
            END
          ) AS FLOAT
        ) > 0
      THEN
        (
          CAST(
            SUM(
              CASE
                WHEN ImplementationFlag = 1 THEN ifnull(SettlementAmount, 0)
              END
            ) AS FLOAT
          )
            / CAST(
              SUM(
                CASE
                  WHEN ImplementationFlag = 1 THEN ifnull(TLCatMigrationDate, 0)
                END
              ) AS FLOAT
            )
        )
      ELSE 0
    END AS PotentialCashRecImplemented,
    CAST(
      SUM(
        CASE
          WHEN ImplementationFlag = 1 THEN ifnull(ScheduledAmount, 0)
        END
      ) AS FLOAT
    ) AS AmountDuetoDateImplemented,
    CAST(
      SUM(distinct
        CASE
          WHEN ifnull(l.L3869Flag, 0) = 0 THEN ifnull(l.MonthlyPayments, 0)
        END
      ) AS FLOAT
    ) AS TotalPaymentsNoL3869,
    CASE
      WHEN
        (CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)) > 0
      THEN
        (
          CAST(SUM(distinct ifnull(l.MonthlyPayments, 0)) AS FLOAT)
            / CAST(SUM(ifnull(TLCatMigrationDate, 0)) AS FLOAT)
        )
      ELSE 0
    END AS RealisedCashRec,
    CAST(
      ifnull(
        SUM(distinct
          CASE
            WHEN ifnull(l.L3869Flag, 0) = 1 THEN ifnull(l.MonthlyPayments, 0)
          END
        ),
        0
      ) AS FLOAT
    ) AS TotalPaymentsL3869,
    SUM(distinct ifnull(l.MonthlyPayments, 0)) AS TotalPayments
  FROM
    LineByLine l
      LEFT JOIN assignment p
        on p.exposurekey = l.ExposureKey
        and p.CDI = l.CDI
        and p.BusinessDate = l.BusinessDate
  GROUP BY
    l.PortfolioName,
    l.BusinessDate
),
Agent AS (
  select
    *
  from
    ca
  union
  select
    *
  from
    fa
  union
  select
    *
  from
    lf
  union
  select
    *
  from
    unassigned
  union
  select
    *
  from
    paidoff
  union
  select
    *
  from
    total
)
SELECT
  BusinessDate,
  RowNumber,
  Channel,
  Team,
  ExternalCompany,
  PortfolioName,
  L3869Flag,
  IFNULL(NumberAssignedBorrowers, 0) AS NumberAssignedBorrowers,
  IFNULL(GBVAssigned, 0) AS GBVAssigned,
  IFNULL(NumberWorkedDeptors, 0) AS NumberWorkedDeptors,
  IFNULL(NumberWorkedLoans, 0) AS NumberWorkedLoans,
  IFNULL(GBVWorked, 0) AS GBVWorked,
  IFNULL(NumberContactedBorrowers, 0) AS NumberContactedBorrowers,
  IFNULL(GBVContacted, 0) AS GBVContacted,
  IFNULL(NumberAppliedBorrowers, 0) AS NumberAppliedBorrowers,
  IFNULL(GBVApplied, 0) AS GBVApplied,
  IFNULL(NumberApporvedBorrowers, 0) AS NumberApporvedBorrowers,
  IFNULL(GBVApproved, 0) AS GBVApproved,
  IFNULL(AverageHaicutApproved, 0) AS AverageHaicutApproved,
  IFNULL(AverageInstallmentsApproved, 0) AS AverageInstallmentsApproved,
  IFNULL(AmounttobePaidApproved, 0) AS AmounttobePaidApproved,
  IFNULL(AmountDuetoDateApproved, 0) AS AmountDuetoDateApproved,
  IFNULL(NumberAppliedBorrowersIntra, 0) AS NumberAppliedBorrowersIntra,
  IFNULL(GBVAppliedIntra, 0) AS GBVAppliedIntra,
  IFNULL(NumberApporvedBorrowersIntra, 0) AS NumberApporvedBorrowersIntra,
  IFNULL(GBVApprovedIntra, 0) AS GBVApprovedIntra,
  IFNULL(AverageHaicutApprovedIntra, 0) AS AverageHaicutApprovedIntra,
  IFNULL(AverageInstallmentsApprovedIntra, 0) AS AverageInstallmentsApprovedIntra,
  IFNULL(AmounttobePaidApprovedIntra, 0) AS AmounttobePaidApprovedIntra,
  IFNULL(AmountDuetoDateApprovedIntra, 0) AS AmountDuetoDateApprovedIntra,
  IFNULL(NumberImplementedBorrowers, 0) AS NumberImplementedBorrowers,
  IFNULL(GBVImplemented, 0) AS GBVImplemented,
  IFNULL(NumberImplementedBorrowersIntra, 0) AS NumberImplementedBorrowersIntra,
  IFNULL(GBVImplementedIntra, 0) AS GBVImplementedIntra,
  IFNULL(AverageHaicutImplemented, 0) AS AverageHaicutImplemented,
  IFNULL(AverageInstallmentsImplemented, 0) AS AverageInstallmentsImplemented,
  IFNULL(AmounttobePaidImplemented, 0) AS AmounttobePaidImplemented,
  IFNULL(PotentialCashRecImplemented, 0) AS PotentialCashRecImplemented,
  IFNULL(AmountDuetoDateImplemented, 0) AS AmountDuetoDateImplemented,
  IFNULL(TotalPaymentsNoL3869, 0) AS TotalPaymentsNoL3869,
  IFNULL(RealisedCashRec, 0) AS RealisedCashRec,
  IFNULL(TotalPaymentsL3869, 0) AS TotalPaymentsL3869,
  IFNULL(TotalPayments, 0) AS TotalPayments
FROM
  Agent

#### FUNCTIONS UPDATE

In [0]:
from pyspark.sql import functions as F, types as T
import re

CATALOG = "p-cdh"
SCHEMA = "primary"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")


def table_exists(ident: str) -> bool:
    """Check if a table/view exists."""
    try:
        return spark._jsparkSession.catalog().tableExists(ident)
    except Exception:
        return False


def parse_desc_rows(rows):
    """Parse DESCRIBE FUNCTION EXTENDED output into a structured dict."""
    lines = []
    for r in rows:
        cand = next((str(v) for v in r if v is not None and str(v).strip() != ""), None)
        if cand:
            lines.append(cand)
    text = "\n".join(lines)

    language = comment = security = None
    null_handling = None
    return_columns, return_type = [], None
    body_sql_raw = None

    # language / comment / security
    for label, var in [("Language", "language"),
                       ("Comment", "comment"),
                       ("Security", "security")]:
        m = re.search(fr"(?im)^\s*{label}:\s*(.+)$", text)
        if m:
            locals()[var] = m.group(1).strip()

    # null handling
    m = re.search(r"(?im)^\s*(Returns null on null input|Called on null input).*", text)
    if m:
        null_handling = m.group(0).strip()

    # return type / columns
    returns_idx = None
    for i, ln in enumerate(lines):
        if re.search(r"(?i)^\s*Returns?:", ln):
            returns_idx = i
            break

    if returns_idx is not None:
        same = lines[returns_idx]

        # inline TABLE<...> format
        mtab = re.search(r"(?is)TABLE\s*<\s*(.*?)\s*>", same)
        if mtab:
            inside = mtab.group(1)
            for part in [x.strip() for x in inside.split(",") if x.strip()]:
                mt = re.match(r"^([A-Za-z_][A-Za-z0-9_]*)\s+(.+)$", part)
                if mt:
                    return_columns.append(
                        {"name": mt.group(1), "type": mt.group(2).strip()}
                    )
            return_type = f"TABLE<{inside}>"
        else:
            # multi-line column list after "Returns:"
            j = returns_idx + 1
            while j < len(lines):
                ln = lines[j].strip()
                if not ln or re.match(
                    r"(?i)^(Language|Owner|Type|Definition|SQL body|Body|Source|Comment|Security):",
                    ln,
                ):
                    break
                mt = re.match(r"^([A-Za-z_][A-Za-z0-9_]*)\s+(.+)$", ln)
                if mt:
                    return_columns.append(
                        {"name": mt.group(1), "type": mt.group(2).strip()}
                    )
                    j += 1
                    continue
                break
            if return_columns:
                inside = ", ".join([f"{c['name']} {c['type']}" for c in return_columns])
                return_type = f"TABLE<{inside}>"
            else:
                m2 = re.search(r"(?im)^\s*Returns?:\s*(.+)$", text)
                if m2:
                    return_type = m2.group(1).strip()

    # raw SQL body
    m = re.search(r"(?im)^\s*(Definition|SQL body|Body|Source):\s*(.*)$", text)
    if m:
        endpos = m.end(0)
        inline = (m.group(2) or "").rstrip()
        rest = text[endpos:]
        body_sql_raw = (
            inline + ("\n" if rest and not rest.startswith("\n") else "") + rest
        ).strip()

    return {
        "language_raw": language,
        "return_columns": return_columns,
        "return_type_raw": return_type,
        "body_sql_raw": body_sql_raw,
        "comment": comment,
        "security": security,
        "null_handling": null_handling,
    }


# ----------------------------------------------------------------------
# Read all functions for the given schema
# ----------------------------------------------------------------------

routines_ident = f"`{CATALOG}`.`information_schema`.`routines`"

if table_exists(routines_ident):
    funcs = spark.sql(f"""
        SELECT specific_name
        FROM {routines_ident}
        WHERE specific_schema = '{SCHEMA}' AND routine_type = 'FUNCTION'
    """)
    func_names = [r.specific_name for r in funcs.collect()]
else:
    funcs = spark.sql(f"SHOW USER FUNCTIONS IN `{CATALOG}`.`{SCHEMA}`").collect()
    func_names = [row["function"].split(".")[-1] for row in funcs]

# ----------------------------------------------------------------------
# Parse DESCRIBE FUNCTION output (metadata, body, raw return info)
# ----------------------------------------------------------------------

rows = []
for name in func_names:
    try:
        desc = spark.sql(
            f"DESCRIBE FUNCTION EXTENDED `{CATALOG}`.`{SCHEMA}`.`{name}`"
        ).collect()
        meta = parse_desc_rows(desc)
        rows.append(
            (
                CATALOG,
                SCHEMA,
                name,
                meta["return_columns"],
                meta["return_type_raw"],
                meta["language_raw"],
                meta["body_sql_raw"],
                meta["comment"],
                meta["security"],
                meta["null_handling"],
            )
        )
    except Exception:
        # ignore functions we can't describe
        pass

schema_desc = T.StructType(
    [
        T.StructField("catalog", T.StringType()),
        T.StructField("schema", T.StringType()),
        T.StructField("function_name", T.StringType()),
        T.StructField(
            "return_columns",
            T.ArrayType(
                T.StructType(
                    [
                        T.StructField("name", T.StringType()),
                        T.StructField("type", T.StringType()),
                    ]
                )
            ),
        ),
        T.StructField("return_type_raw", T.StringType()),
        T.StructField("language_raw", T.StringType()),
        T.StructField("body_sql_raw", T.StringType()),
        T.StructField("comment", T.StringType()),
        T.StructField("security", T.StringType()),
        T.StructField("null_handling", T.StringType()),
    ]
)

df = spark.createDataFrame(rows, schema=schema_desc)

# ----------------------------------------------------------------------
# Get full parameter info from INFORMATION_SCHEMA.PARAMETERS
#   (preserving original parameter order)
#   *** FIX: use FULL_DATA_TYPE to keep DECIMAL(19,2) etc. ***
# ----------------------------------------------------------------------

param_ident = f"`{CATALOG}`.`information_schema`.`parameters`"
params_df = spark.sql(f"""
    SELECT
        specific_name  AS function_name,
        ordinal_position,
        parameter_name AS name,
        COALESCE(full_data_type, data_type) AS type
    FROM {param_ident}
    WHERE specific_schema = '{SCHEMA}'
""")

# Preserve parameter order using ordinal_position
params_grouped = (
    params_df
    .groupBy("function_name")
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct("ordinal_position", "name", "type")
            )
        ).alias("parameters_ord")
    )
    .withColumn(
        "parameters",
        F.expr(
            "transform(parameters_ord, x -> named_struct('name', x.name, 'type', x.type))"
        )
    )
    .drop("parameters_ord")
)

df = (
    df.join(params_grouped, "function_name", "left")
      .withColumn("language_value", F.upper(F.coalesce("language_raw", F.lit("SQL"))))
)

# Build "name type" parameter list for CREATE FUNCTION
df = df.withColumn(
    "param_list_sql",
    F.when(
        F.size("parameters") > 0,
        F.expr(
            "array_join(transform(parameters, x -> concat(x.name, ' ', x.type)), ', ')"
        ),
    ).otherwise(F.lit("")),
)

# Dummy argument values to call the function safely
df = df.withColumn(
    "param_call_args_sql",
    F.when(
        F.size("parameters") > 0,
        F.expr(
            """
          array_join(
            transform(
              parameters,
              x -> CASE
                     WHEN upper(x.type) LIKE 'DATE%' THEN concat("DATE '", '2000-01-01', "'")
                     WHEN upper(x.type) LIKE 'TIMESTAMP%' THEN concat("TIMESTAMP '", '2000-01-01 00:00:00', "'")
                     WHEN upper(x.type) LIKE 'STRING%' OR upper(x.type) LIKE 'VARCHAR%' THEN "'X'"
                     WHEN upper(x.type) LIKE 'INT%' OR upper(x.type) LIKE 'BIGINT%' 
                          OR upper(x.type) LIKE 'SMALLINT%' OR upper(x.type) LIKE 'TINYINT%' THEN '0'
                     WHEN upper(x.type) LIKE 'DECIMAL%' OR upper(x.type) LIKE 'NUMERIC%' 
                          OR upper(x.type) LIKE 'DOUBLE%' OR upper(x.type) LIKE 'FLOAT%' THEN '0.0'
                     WHEN upper(x.type) LIKE 'BOOLEAN%' THEN 'FALSE'
                     ELSE 'NULL'
                   END
            ),
            ', '
          )
        """
        ),
    ).otherwise(F.lit("")),
)

# ----------------------------------------------------------------------
# Collect grants per function
# ----------------------------------------------------------------------

grants_df = None
for name in func_names:
    try:
        g = (
            spark.sql(f"SHOW GRANTS ON FUNCTION `{CATALOG}`.`{SCHEMA}`.`{name}`")
            .withColumn("catalog", F.lit(CATALOG))
            .withColumn("schema", F.lit(SCHEMA))
            .withColumn("function_name", F.lit(name))
        )
        grants_df = g if grants_df is None else grants_df.unionByName(
            g, allowMissingColumns=True
        )
    except Exception:
        # no grants or not visible
        pass

if grants_df is None:
    grants_ready = spark.createDataFrame(
        [], "catalog STRING, schema STRING, function_name STRING, grant_sql ARRAY<STRING>"
    )
else:
    cols = {c.lower(): c for c in grants_df.columns}
    pcol = cols.get("principal") or cols.get("grantee")
    acol = cols.get("actiontype") or cols.get("privilege") or cols.get("action")
    if pcol and acol:
        fqn_for_grant = F.concat(
            F.lit("`"),
            F.col("catalog"),
            F.lit("`.`"),
            F.col("schema"),
            F.lit("`.`"),
            F.col("function_name"),
            F.lit("`"),
        )
        grants_ready = (
            grants_df.select(
                "catalog",
                "schema",
                "function_name",
                F.col(pcol).alias("grantee"),
                F.col(acol).alias("privilege"),
            )
            .where(F.col("grantee").isNotNull() & F.col("privilege").isNotNull())
            .withColumn(
                "grant_sql_single",
                F.concat(
                    F.lit("GRANT "),
                    F.col("privilege"),
                    F.lit(" ON FUNCTION "),
                    fqn_for_grant,
                    F.lit(" TO `"),
                    F.col("grantee"),
                    F.lit("`"),
                ),
            )
            .groupBy("catalog", "schema", "function_name")
            .agg(F.collect_list("grant_sql_single").alias("grant_sql"))
        )
    else:
        grants_ready = spark.createDataFrame(
            [], "catalog STRING, schema STRING, function_name STRING, grant_sql ARRAY<STRING>"
        )

# ----------------------------------------------------------------------
# Try to infer TABLE return schemas by actually running the functions
# ----------------------------------------------------------------------

infer_data = []
for r in df.collect():
    inferred_cols = []
    if (r.return_type_raw or "").upper().startswith("TABLE") and r.body_sql_raw:
        # Prepare the SQL body, replacing parameter names with dummy values
        sql_body = r.body_sql_raw
        params = r.parameters or []
        dummy_args = (r.param_call_args_sql or "").split(", ")
        param_map = {p['name']: v for p, v in zip(params, dummy_args)}
        # Replace parameter references in the SQL body with dummy values
        for pname, pval in param_map.items():
            # Replace only whole word matches (avoid partial replacements)
            sql_body = re.sub(rf"\b{re.escape(pname)}\b", pval, sql_body)
        # Wrap in SELECT * FROM (...) LIMIT 0 if not already a SELECT
        if not sql_body.strip().upper().startswith("SELECT"):
            sql_body = f"SELECT * FROM ({sql_body})"
        q = f"{sql_body} LIMIT 0"
        try:
            sdf = spark.sql(q)
            inferred_cols = [
                {"name": f.name, "type": f.dataType.simpleString().upper()}
                for f in sdf.schema.fields
            ]
        except Exception:
            inferred_cols = []
    infer_data.append((r.catalog, r.schema, r.function_name, inferred_cols))

infer_df = spark.createDataFrame(
    infer_data,
    """
    catalog STRING,
    schema STRING,
    function_name STRING,
    inferred_return_columns ARRAY<STRUCT<name:STRING,type:STRING>>
    """,
).withColumn(
    "inferred_return_type_full",
    F.when(
        F.size("inferred_return_columns") > 0,
        F.concat(
            F.lit("TABLE ("),
            F.expr(
                "array_join(transform(inferred_return_columns, x -> concat(x.name,' ',x.type)), ', ')"
            ),
            F.lit(")"),
        ),
    ),
)

# ----------------------------------------------------------------------
# Build final definition + grants per function
# ----------------------------------------------------------------------

res = (
    df.alias("d")
    .join(grants_ready.alias("g"), ["catalog", "schema", "function_name"], "left")
    .join(infer_df.alias("i"), ["catalog", "schema", "function_name"], "left")
    .withColumn(
        "effective_return_type",
        F.coalesce(
            F.col("i.inferred_return_type_full"),
            # handle old TABLE<...> syntax
            F.regexp_replace(
                F.regexp_replace(F.col("d.return_type_raw"), "<", "("),
                ">",
                ")",
            ),
        ),
    )
    .withColumn(
        "create_function_sql",
        F.concat(
            F.lit("CREATE OR REPLACE FUNCTION `"),
            F.col("catalog"),
            F.lit("`.`"),
            F.col("schema"),
            F.lit("`.`"),
            F.col("function_name"),
            F.lit("`("),
            F.col("param_list_sql"),
            F.lit(") RETURNS "),
            F.col("effective_return_type"),
            F.lit(" LANGUAGE "),
            F.col("language_value"),
            F.lit(" RETURN "),
            F.col("body_sql_raw"),
            F.lit(";"),
        ),
    )
    .withColumn(
        "grant_sql",
        F.coalesce(
            F.col("g.grant_sql"),
            F.array().cast(T.ArrayType(T.StringType())),
        ),
    )
    .select(
        "catalog",
        "schema",
        "function_name",
        "param_list_sql",
        "parameters",
        "param_call_args_sql",
        "effective_return_type",
        "inferred_return_columns",
        "body_sql_raw",
        "language_value",
        "comment",
        "security",
        "null_handling",
        "grant_sql",
        "create_function_sql",
    )
)

# ----------------------------------------------------------------------
# Apply CREATE FUNCTION + GRANTS and capture status
# ----------------------------------------------------------------------

rows_to_apply = res.select(
    "catalog", "schema", "function_name", "create_function_sql", "grant_sql"
).collect()

status_records = []

for r in rows_to_apply:
    fn_fqn = f"`{r.catalog}`.`{r.schema}`.`{r.function_name}`"

    create_status = "SKIPPED"
    create_error = None
    grant_status = "SKIPPED"
    grant_error = None

    # create / replace function
    create_sql = (r.create_function_sql or "").strip()
    if create_sql:
        create_sql_clean = create_sql.rstrip().rstrip(";")
        try:
            spark.sql(create_sql_clean)
            create_status = "SUCCESS"
        except Exception as e:
            create_status = "FAILED"
            create_error = str(e)

    # only run grants if create succeeded
    grants = r.grant_sql or []
    if grants:
        if create_status != "SUCCESS":
            grant_status = "SKIPPED"
        else:
            all_ok = True
            errors = []
            for g in grants:
                if not g:
                    continue
                grant_sql_clean = g.rstrip().rstrip(";")
                try:
                    spark.sql(grant_sql_clean)
                except Exception as e:
                    all_ok = False
                    errors.append(str(e))
            grant_status = "SUCCESS" if all_ok else "FAILED"
            if errors:
                grant_error = " | ".join(errors)[:4000]

    status_records.append(
        (
            r.catalog,
            r.schema,
            r.function_name,
            create_status,
            create_error,
            grant_status,
            grant_error,
        )
    )

status_schema = T.StructType(
    [
        T.StructField("catalog", T.StringType()),
        T.StructField("schema", T.StringType()),
        T.StructField("function_name", T.StringType()),
        T.StructField("create_status", T.StringType()),
        T.StructField("create_error", T.StringType()),
        T.StructField("grant_status", T.StringType()),
        T.StructField("grant_error", T.StringType()),
    ]
)

status_df = spark.createDataFrame(status_records, schema=status_schema)

# final result with statuses
res_with_status = res.join(status_df, ["catalog", "schema", "function_name"], "left")

display(res_with_status)


In [0]:
from pyspark.sql import functions as F, types as T
import re

CATALOG = "p-cdh"
SCHEMA = "derivative"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")


def table_exists(ident: str) -> bool:
    """Check if a table/view exists."""
    try:
        return spark._jsparkSession.catalog().tableExists(ident)
    except Exception:
        return False


def parse_desc_rows(rows):
    """Parse DESCRIBE FUNCTION EXTENDED output into a structured dict."""
    lines = []
    for r in rows:
        cand = next((str(v) for v in r if v is not None and str(v).strip() != ""), None)
        if cand:
            lines.append(cand)
    text = "\n".join(lines)

    language = comment = security = None
    null_handling = None
    return_columns, return_type = [], None
    body_sql_raw = None

    # language / comment / security
    for label, var in [("Language", "language"),
                       ("Comment", "comment"),
                       ("Security", "security")]:
        m = re.search(fr"(?im)^\s*{label}:\s*(.+)$", text)
        if m:
            locals()[var] = m.group(1).strip()

    # null handling
    m = re.search(r"(?im)^\s*(Returns null on null input|Called on null input).*", text)
    if m:
        null_handling = m.group(0).strip()

    # return type / columns
    returns_idx = None
    for i, ln in enumerate(lines):
        if re.search(r"(?i)^\s*Returns?:", ln):
            returns_idx = i
            break

    if returns_idx is not None:
        same = lines[returns_idx]

        # inline TABLE<...> format
        mtab = re.search(r"(?is)TABLE\s*<\s*(.*?)\s*>", same)
        if mtab:
            inside = mtab.group(1)
            for part in [x.strip() for x in inside.split(",") if x.strip()]:
                mt = re.match(r"^([A-Za-z_][A-Za-z0-9_]*)\s+(.+)$", part)
                if mt:
                    return_columns.append(
                        {"name": mt.group(1), "type": mt.group(2).strip()}
                    )
            return_type = f"TABLE<{inside}>"
        else:
            # multi-line column list after "Returns:"
            j = returns_idx + 1
            while j < len(lines):
                ln = lines[j].strip()
                if not ln or re.match(
                    r"(?i)^(Language|Owner|Type|Definition|SQL body|Body|Source|Comment|Security):",
                    ln,
                ):
                    break
                mt = re.match(r"^([A-Za-z_][A-Za-z0-9_]*)\s+(.+)$", ln)
                if mt:
                    return_columns.append(
                        {"name": mt.group(1), "type": mt.group(2).strip()}
                    )
                    j += 1
                    continue
                break
            if return_columns:
                inside = ", ".join([f"{c['name']} {c['type']}" for c in return_columns])
                return_type = f"TABLE<{inside}>"
            else:
                m2 = re.search(r"(?im)^\s*Returns?:\s*(.+)$", text)
                if m2:
                    return_type = m2.group(1).strip()

    # raw SQL body
    m = re.search(r"(?im)^\s*(Definition|SQL body|Body|Source):\s*(.*)$", text)
    if m:
        endpos = m.end(0)
        inline = (m.group(2) or "").rstrip()
        rest = text[endpos:]
        body_sql_raw = (
            inline + ("\n" if rest and not rest.startswith("\n") else "") + rest
        ).strip()

    return {
        "language_raw": language,
        "return_columns": return_columns,
        "return_type_raw": return_type,
        "body_sql_raw": body_sql_raw,
        "comment": comment,
        "security": security,
        "null_handling": null_handling,
    }


# ----------------------------------------------------------------------
# Read all functions for the given schema
# ----------------------------------------------------------------------

routines_ident = f"`{CATALOG}`.`information_schema`.`routines`"

if table_exists(routines_ident):
    funcs = spark.sql(f"""
        SELECT specific_name
        FROM {routines_ident}
        WHERE specific_schema = '{SCHEMA}' AND routine_type = 'FUNCTION'
    """)
    func_names = [r.specific_name for r in funcs.collect()]
else:
    funcs = spark.sql(f"SHOW USER FUNCTIONS IN `{CATALOG}`.`{SCHEMA}`").collect()
    func_names = [row["function"].split(".")[-1] for row in funcs]

# ----------------------------------------------------------------------
# Parse DESCRIBE FUNCTION output (metadata, body, raw return info)
# ----------------------------------------------------------------------

rows = []
for name in func_names:
    try:
        desc = spark.sql(
            f"DESCRIBE FUNCTION EXTENDED `{CATALOG}`.`{SCHEMA}`.`{name}`"
        ).collect()
        meta = parse_desc_rows(desc)
        rows.append(
            (
                CATALOG,
                SCHEMA,
                name,
                meta["return_columns"],
                meta["return_type_raw"],
                meta["language_raw"],
                meta["body_sql_raw"],
                meta["comment"],
                meta["security"],
                meta["null_handling"],
            )
        )
    except Exception:
        # ignore functions we can't describe
        pass

schema_desc = T.StructType(
    [
        T.StructField("catalog", T.StringType()),
        T.StructField("schema", T.StringType()),
        T.StructField("function_name", T.StringType()),
        T.StructField(
            "return_columns",
            T.ArrayType(
                T.StructType(
                    [
                        T.StructField("name", T.StringType()),
                        T.StructField("type", T.StringType()),
                    ]
                )
            ),
        ),
        T.StructField("return_type_raw", T.StringType()),
        T.StructField("language_raw", T.StringType()),
        T.StructField("body_sql_raw", T.StringType()),
        T.StructField("comment", T.StringType()),
        T.StructField("security", T.StringType()),
        T.StructField("null_handling", T.StringType()),
    ]
)

df = spark.createDataFrame(rows, schema=schema_desc)

# ----------------------------------------------------------------------
# Get full parameter info from INFORMATION_SCHEMA.PARAMETERS
#   (preserving original parameter order)
#   *** FIX: use FULL_DATA_TYPE to keep DECIMAL(19,2) etc. ***
# ----------------------------------------------------------------------

param_ident = f"`{CATALOG}`.`information_schema`.`parameters`"
params_df = spark.sql(f"""
    SELECT
        specific_name  AS function_name,
        ordinal_position,
        parameter_name AS name,
        COALESCE(full_data_type, data_type) AS type
    FROM {param_ident}
    WHERE specific_schema = '{SCHEMA}'
""")

# Preserve parameter order using ordinal_position
params_grouped = (
    params_df
    .groupBy("function_name")
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct("ordinal_position", "name", "type")
            )
        ).alias("parameters_ord")
    )
    .withColumn(
        "parameters",
        F.expr(
            "transform(parameters_ord, x -> named_struct('name', x.name, 'type', x.type))"
        )
    )
    .drop("parameters_ord")
)

df = (
    df.join(params_grouped, "function_name", "left")
      .withColumn("language_value", F.upper(F.coalesce("language_raw", F.lit("SQL"))))
)

# Build "name type" parameter list for CREATE FUNCTION
df = df.withColumn(
    "param_list_sql",
    F.when(
        F.size("parameters") > 0,
        F.expr(
            "array_join(transform(parameters, x -> concat(x.name, ' ', x.type)), ', ')"
        ),
    ).otherwise(F.lit("")),
)

# Dummy argument values to call the function safely
df = df.withColumn(
    "param_call_args_sql",
    F.when(
        F.size("parameters") > 0,
        F.expr(
            """
          array_join(
            transform(
              parameters,
              x -> CASE
                     WHEN upper(x.type) LIKE 'DATE%' THEN concat("DATE '", '2000-01-01', "'")
                     WHEN upper(x.type) LIKE 'TIMESTAMP%' THEN concat("TIMESTAMP '", '2000-01-01 00:00:00', "'")
                     WHEN upper(x.type) LIKE 'STRING%' OR upper(x.type) LIKE 'VARCHAR%' THEN "'X'"
                     WHEN upper(x.type) LIKE 'INT%' OR upper(x.type) LIKE 'BIGINT%' 
                          OR upper(x.type) LIKE 'SMALLINT%' OR upper(x.type) LIKE 'TINYINT%' THEN '0'
                     WHEN upper(x.type) LIKE 'DECIMAL%' OR upper(x.type) LIKE 'NUMERIC%' 
                          OR upper(x.type) LIKE 'DOUBLE%' OR upper(x.type) LIKE 'FLOAT%' THEN '0.0'
                     WHEN upper(x.type) LIKE 'BOOLEAN%' THEN 'FALSE'
                     ELSE 'NULL'
                   END
            ),
            ', '
          )
        """
        ),
    ).otherwise(F.lit("")),
)

# ----------------------------------------------------------------------
# Collect grants per function
# ----------------------------------------------------------------------

grants_df = None
for name in func_names:
    try:
        g = (
            spark.sql(f"SHOW GRANTS ON FUNCTION `{CATALOG}`.`{SCHEMA}`.`{name}`")
            .withColumn("catalog", F.lit(CATALOG))
            .withColumn("schema", F.lit(SCHEMA))
            .withColumn("function_name", F.lit(name))
        )
        grants_df = g if grants_df is None else grants_df.unionByName(
            g, allowMissingColumns=True
        )
    except Exception:
        # no grants or not visible
        pass

if grants_df is None:
    grants_ready = spark.createDataFrame(
        [], "catalog STRING, schema STRING, function_name STRING, grant_sql ARRAY<STRING>"
    )
else:
    cols = {c.lower(): c for c in grants_df.columns}
    pcol = cols.get("principal") or cols.get("grantee")
    acol = cols.get("actiontype") or cols.get("privilege") or cols.get("action")
    if pcol and acol:
        fqn_for_grant = F.concat(
            F.lit("`"),
            F.col("catalog"),
            F.lit("`.`"),
            F.col("schema"),
            F.lit("`.`"),
            F.col("function_name"),
            F.lit("`"),
        )
        grants_ready = (
            grants_df.select(
                "catalog",
                "schema",
                "function_name",
                F.col(pcol).alias("grantee"),
                F.col(acol).alias("privilege"),
            )
            .where(F.col("grantee").isNotNull() & F.col("privilege").isNotNull())
            .withColumn(
                "grant_sql_single",
                F.concat(
                    F.lit("GRANT "),
                    F.col("privilege"),
                    F.lit(" ON FUNCTION "),
                    fqn_for_grant,
                    F.lit(" TO `"),
                    F.col("grantee"),
                    F.lit("`"),
                ),
            )
            .groupBy("catalog", "schema", "function_name")
            .agg(F.collect_list("grant_sql_single").alias("grant_sql"))
        )
    else:
        grants_ready = spark.createDataFrame(
            [], "catalog STRING, schema STRING, function_name STRING, grant_sql ARRAY<STRING>"
        )

# ----------------------------------------------------------------------
# Try to infer TABLE return schemas by actually running the functions
# ----------------------------------------------------------------------

infer_data = []
for r in df.collect():
    inferred_cols = []
    if (r.return_type_raw or "").upper().startswith("TABLE") and r.body_sql_raw:
        # Prepare the SQL body, replacing parameter names with dummy values
        sql_body = r.body_sql_raw
        params = r.parameters or []
        dummy_args = (r.param_call_args_sql or "").split(", ")
        param_map = {p['name']: v for p, v in zip(params, dummy_args)}
        # Replace parameter references in the SQL body with dummy values
        for pname, pval in param_map.items():
            # Replace only whole word matches (avoid partial replacements)
            sql_body = re.sub(rf"\b{re.escape(pname)}\b", pval, sql_body)
        # Wrap in SELECT * FROM (...) LIMIT 0 if not already a SELECT
        if not sql_body.strip().upper().startswith("SELECT"):
            sql_body = f"SELECT * FROM ({sql_body})"
        q = f"{sql_body} LIMIT 0"
        try:
            sdf = spark.sql(q)
            inferred_cols = [
                {"name": f.name, "type": f.dataType.simpleString().upper()}
                for f in sdf.schema.fields
            ]
        except Exception:
            inferred_cols = []
    infer_data.append((r.catalog, r.schema, r.function_name, inferred_cols))

infer_df = spark.createDataFrame(
    infer_data,
    """
    catalog STRING,
    schema STRING,
    function_name STRING,
    inferred_return_columns ARRAY<STRUCT<name:STRING,type:STRING>>
    """,
).withColumn(
    "inferred_return_type_full",
    F.when(
        F.size("inferred_return_columns") > 0,
        F.concat(
            F.lit("TABLE ("),
            F.expr(
                "array_join(transform(inferred_return_columns, x -> concat(x.name,' ',x.type)), ', ')"
            ),
            F.lit(")"),
        ),
    ),
)

# ----------------------------------------------------------------------
# Build final definition + grants per function
# ----------------------------------------------------------------------

res = (
    df.alias("d")
    .join(grants_ready.alias("g"), ["catalog", "schema", "function_name"], "left")
    .join(infer_df.alias("i"), ["catalog", "schema", "function_name"], "left")
    .withColumn(
        "effective_return_type",
        F.coalesce(
            F.col("i.inferred_return_type_full"),
            # handle old TABLE<...> syntax
            F.regexp_replace(
                F.regexp_replace(F.col("d.return_type_raw"), "<", "("),
                ">",
                ")",
            ),
        ),
    )
    .withColumn(
        "create_function_sql",
        F.concat(
            F.lit("CREATE OR REPLACE FUNCTION `"),
            F.col("catalog"),
            F.lit("`.`"),
            F.col("schema"),
            F.lit("`.`"),
            F.col("function_name"),
            F.lit("`("),
            F.col("param_list_sql"),
            F.lit(") RETURNS "),
            F.col("effective_return_type"),
            F.lit(" LANGUAGE "),
            F.col("language_value"),
            F.lit(" RETURN "),
            F.col("body_sql_raw"),
            F.lit(";"),
        ),
    )
    .withColumn(
        "grant_sql",
        F.coalesce(
            F.col("g.grant_sql"),
            F.array().cast(T.ArrayType(T.StringType())),
        ),
    )
    .select(
        "catalog",
        "schema",
        "function_name",
        "param_list_sql",
        "parameters",
        "param_call_args_sql",
        "effective_return_type",
        "inferred_return_columns",
        "body_sql_raw",
        "language_value",
        "comment",
        "security",
        "null_handling",
        "grant_sql",
        "create_function_sql",
    )
)

# ----------------------------------------------------------------------
# Apply CREATE FUNCTION + GRANTS and capture status
# ----------------------------------------------------------------------

rows_to_apply = res.select(
    "catalog", "schema", "function_name", "create_function_sql", "grant_sql"
).collect()

status_records = []

for r in rows_to_apply:
    fn_fqn = f"`{r.catalog}`.`{r.schema}`.`{r.function_name}`"

    create_status = "SKIPPED"
    create_error = None
    grant_status = "SKIPPED"
    grant_error = None

    # create / replace function
    create_sql = (r.create_function_sql or "").strip()
    if create_sql:
        create_sql_clean = create_sql.rstrip().rstrip(";")
        try:
            spark.sql(create_sql_clean)
            create_status = "SUCCESS"
        except Exception as e:
            create_status = "FAILED"
            create_error = str(e)

    # only run grants if create succeeded
    grants = r.grant_sql or []
    if grants:
        if create_status != "SUCCESS":
            grant_status = "SKIPPED"
        else:
            all_ok = True
            errors = []
            for g in grants:
                if not g:
                    continue
                grant_sql_clean = g.rstrip().rstrip(";")
                try:
                    spark.sql(grant_sql_clean)
                except Exception as e:
                    all_ok = False
                    errors.append(str(e))
            grant_status = "SUCCESS" if all_ok else "FAILED"
            if errors:
                grant_error = " | ".join(errors)[:4000]

    status_records.append(
        (
            r.catalog,
            r.schema,
            r.function_name,
            create_status,
            create_error,
            grant_status,
            grant_error,
        )
    )

status_schema = T.StructType(
    [
        T.StructField("catalog", T.StringType()),
        T.StructField("schema", T.StringType()),
        T.StructField("function_name", T.StringType()),
        T.StructField("create_status", T.StringType()),
        T.StructField("create_error", T.StringType()),
        T.StructField("grant_status", T.StringType()),
        T.StructField("grant_error", T.StringType()),
    ]
)

status_df = spark.createDataFrame(status_records, schema=status_schema)

# final result with statuses
res_with_status = res.join(status_df, ["catalog", "schema", "function_name"], "left")

display(res_with_status)
